# SOL TDI Band Hook Bot — Final Config
## Strategy summary
- **Timeframe:** 15M SOL/USDT perpetual
- **Entry:** RSI crosses back inside the band (long: crosses above lower band / short: crosses below upper band)
- **Exit:** Immediate — exit on the same bar the opposite band cross fires (trail = 0)
- **Sizing:** 100% equity normally, 300% when RSI ≤ 20 (long) or ≥ 80 (short)
- **ADX filter:** Skip new entries when ADX > 30 (trending market)
- **Circuit breaker:** Pause rest of month if monthly loss > $125
- **Backtest result:** +1,850% over 2.5 years, 33.63% max DD, 60.3% WR, CAGR ~228%

## Setup
1. Run the install cell
2. Set `STARTING_CAPITAL` in config
3. Run all cells top to bottom
4. Bot polls every 15 minutes — check `sol_bot_final.log` and `sol_bot_final_state.json`

## Going live
Set `PAPER_TRADING = False` and configure `HL_PRIVATE_KEY`


In [1]:
!pip install requests pandas numpy schedule

In [2]:
import requests
import pandas as pd
import numpy as np
import json
import time
import logging
import schedule
from datetime import datetime, timezone
from pathlib import Path

## ⚙️ Config — edit before running

In [3]:
# ── Mode ──────────────────────────────────────────
PAPER_TRADING    = True

# ── Capital ───────────────────────────────────────
STARTING_CAPITAL = 1000.0     # USD

# ── TDI ───────────────────────────────────────────
RSI_PERIOD  = 10
BAND_LENGTH = 34
FAST_MA     = 7
SLOW_MA     = 2
BB_MULT     = 2.5

# ── Trail exit ────────────────────────────────────
TRAIL_PULLBACK = 0.0

# ── Position sizing ───────────────────────────────
BASE_MULT        = 1.0        # 100% equity — normal
TRIPLE_MULT      = 3.0        # 300% equity — extreme RSI (3x leverage)
RSI_TRIPLE_LONG  = 20         # RSI <= this → 3x long
RSI_TRIPLE_SHORT = 80         # RSI >= this → 3x short

# ── ADX regime filter ─────────────────────────────
ADX_PERIOD       = 14
ADX_TREND_THRESH = 30         # skip new entries when ADX > this
                               # ADX > 40 = extreme trend, mean reversion fails

# ── Circuit breaker ───────────────────────────────
MONTHLY_LOSS_LIMIT = 125.0    # USD/month — None to disable

# ── Symbols ───────────────────────────────────────
SYMBOL      = 'SOLUSDT'       # Binance (data)
HL_SYMBOL   = 'SOL'           # Hyperliquid (execution)
INTERVAL    = '15m'
LOOKBACK    = 500             # candles per poll (500 = ~5 days, enough for warmup)

# ── Files ─────────────────────────────────────────
LOG_FILE   = 'sol_bot_final.log'
STATE_FILE = 'sol_bot_final_state.json'

# ── Hyperliquid (live only) ────────────────────────
HL_PRIVATE_KEY = ''
HL_WALLET      = ''

print(f"Mode:            {'📄 PAPER' if PAPER_TRADING else '🔴 LIVE'}")
print(f"Capital:         ${STARTING_CAPITAL:,.2f}")
print(f"Trail:           {TRAIL_PULLBACK} RSI pts")
print(f"Sizing:          {BASE_MULT*100:.0f}% normal | {TRIPLE_MULT*100:.0f}% when RSI<={RSI_TRIPLE_LONG} or >={RSI_TRIPLE_SHORT}")
print(f"ADX filter:      skip entries when ADX > {ADX_TREND_THRESH}")
print(f"Circuit breaker: ${MONTHLY_LOSS_LIMIT}/mo" if MONTHLY_LOSS_LIMIT else "Circuit breaker: disabled")

Mode:            📄 PAPER
Capital:         $1,000.00
Trail:           0.0 RSI pts
Sizing:          100% normal | 300% when RSI<=20 or >=80
ADX filter:      skip entries when ADX > 30
Circuit breaker: $125.0/mo


## 📝 Logger + State

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('sol_bot_final')

def load_state():
    if Path(STATE_FILE).exists():
        with open(STATE_FILE) as f:
            s = json.load(f)
        log.info(f"State loaded | equity: ${s['equity']:.2f} | "
                 f"position: {s['position']} | month PnL: ${s['month_pnl']:.2f}")
        return s
    log.info("No state file — starting fresh")
    return {
        'equity':          STARTING_CAPITAL,
        'peak_equity':     STARTING_CAPITAL,
        'position':        None,
        'entry_price':     None,
        'entry_rsi':       None,
        'entry_mult':      None,
        'hook_rsi_level':  None,
        'waiting_trail':   False,
        'month_pnl':       0.0,
        'current_month':   None,
        'breaker_on':      False,
        'total_trades':    0,
        'wins':            0,
        'losses':          0,
        'trades':          [],
    }

def save_state(s):
    with open(STATE_FILE, 'w') as f:
        json.dump(s, f, indent=2, default=str)

state = load_state()
log.info("Logger and state ready")

2026-05-19 17:49:53 | INFO | No state file — starting fresh
2026-05-19 17:49:53 | INFO | Logger and state ready


## 📐 TDI + ADX calculation

In [5]:
def fetch_candles(symbol, interval, limit=500):
    r = requests.get('https://api.binance.com/api/v3/klines',
        params={'symbol':symbol,'interval':interval,'limit':limit}, timeout=10)
    r.raise_for_status()
    df = pd.DataFrame(r.json(), columns=[
        'open_time','open','high','low','close','volume',
        'close_time','qv','trades','tb','tq','ignore'])
    df['ts']    = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    df['close'] = df['close'].astype(float)
    df['high']  = df['high'].astype(float)
    df['low']   = df['low'].astype(float)
    return df[['ts','high','low','close']].reset_index(drop=True)

def calc_rsi(close, period):
    """Wilder's RSI — matches Pine Script exactly."""
    delta = close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_g = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_l = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs    = avg_g / avg_l.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def calc_adx(df, period=14):
    """Wilder's ADX — matches Pine Script ta.adx() exactly."""
    hi, lo, cl = df['high'], df['low'], df['close']
    tr = pd.concat([hi-lo, (hi-cl.shift(1)).abs(), (lo-cl.shift(1)).abs()], axis=1).max(axis=1)
    up   = hi - hi.shift(1)
    down = lo.shift(1) - lo
    pdm  = pd.Series(np.where((up>down)&(up>0),   up,   0.0), index=df.index)
    mdm  = pd.Series(np.where((down>up)&(down>0), down, 0.0), index=df.index)
    atr  = tr.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    pdi  = 100 * pdm.ewm(alpha=1/period, min_periods=period, adjust=False).mean() / atr
    mdi  = 100 * mdm.ewm(alpha=1/period, min_periods=period, adjust=False).mean() / atr
    dx   = 100 * (pdi-mdi).abs() / (pdi+mdi).replace(0, np.nan)
    adx  = dx.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    return adx, pdi, mdi

def calc_tdi(df):
    df = df.copy()
    df['r']       = calc_rsi(df['close'], RSI_PERIOD)
    df['ma']      = df['r'].rolling(BAND_LENGTH).mean()
    df['std']     = df['r'].rolling(BAND_LENGTH).std(ddof=0)
    df['up']      = df['ma'] + BB_MULT * df['std']
    df['dn']      = df['ma'] - BB_MULT * df['std']
    df['mbb']     = df['r'].rolling(FAST_MA).mean()
    df['mab']     = df['r'].rolling(SLOW_MA).mean()
    df['r_prev']  = df['r'].shift(1)
    df['up_prev'] = df['up'].shift(1)
    df['dn_prev'] = df['dn'].shift(1)
    # Hook fires when RSI crosses BACK INSIDE the band (matches TradingView visual)
    # upper hook down: prev bar RSI was above upper band, current bar RSI crossed below upper band
    # lower hook up:   prev bar RSI was below lower band, current bar RSI crossed above lower band
    df['upper_hook_down'] = (df['r_prev'] >= df['up_prev']) & (df['r'] < df['up'])
    df['lower_hook_up']   = (df['r_prev'] <= df['dn_prev']) & (df['r'] > df['dn'])
    df['adx'], df['pdi'], df['mdi'] = calc_adx(df, ADX_PERIOD)
    df['trending'] = df['adx'] > ADX_TREND_THRESH
    return df

log.info("TDI + ADX functions ready")

2026-05-19 17:49:55 | INFO | TDI + ADX functions ready


## 🔌 Order execution

In [6]:
def get_multiplier(direction, entry_rsi):
    if direction == 'long'  and entry_rsi <= RSI_TRIPLE_LONG:  return TRIPLE_MULT
    if direction == 'short' and entry_rsi >= RSI_TRIPLE_SHORT: return TRIPLE_MULT
    return BASE_MULT

def execute_order(direction, notional, price):
    if PAPER_TRADING:
        log.info(f"    [PAPER] {direction.upper()} SOL | notional: ${notional:.2f} | price: ${price:.4f}")
        return {'status': 'paper'}
    # ── Live Hyperliquid ──────────────────────────────────────────
    # from hyperliquid.exchange import Exchange
    # from hyperliquid.utils import constants
    # import eth_account
    # account  = eth_account.Account.from_key(HL_PRIVATE_KEY)
    # exchange = Exchange(account, constants.MAINNET_API_URL)
    # qty      = round(notional / price, 2)
    # result   = exchange.market_open(HL_SYMBOL, direction=='long', qty)
    # log.info(f"    [LIVE] {result}")
    # return result
    raise NotImplementedError("Set PAPER_TRADING=False and configure HL_PRIVATE_KEY")

def close_order(direction, price):
    if PAPER_TRADING:
        log.info(f"    [PAPER] CLOSE {direction.upper()} SOL | price: ${price:.4f}")
        return {'status': 'paper'}
    # exchange.market_close(HL_SYMBOL)
    raise NotImplementedError("Configure live execution first")

log.info("Execution functions ready")

2026-05-19 17:49:56 | INFO | Execution functions ready


## 🔁 Core strategy loop

In [7]:
def run_once():
    global state
    now = datetime.now(timezone.utc)
    log.info(f"── Poll {now.strftime('%Y-%m-%d %H:%M')} UTC ──")

    # ── Fetch + calculate ──
    try:
        df  = fetch_candles(SYMBOL, INTERVAL, LOOKBACK)
        tdi = calc_tdi(df)
    except Exception as e:
        log.error(f"Data fetch failed: {e}")
        return

    # Use second-to-last candle (last is still forming)
    row      = tdi.iloc[-2]
    price    = row['close']
    rsi      = row['r']
    r_prev   = row['r_prev']
    hook_up  = bool(row['lower_hook_up'])
    hook_dn  = bool(row['upper_hook_down'])
    adx_val  = row['adx']
    trending = bool(row['trending'])

    log.info(f"  RSI: {rsi:.2f} | upper: {row['up']:.2f} | lower: {row['dn']:.2f} | "
             f"price: ${price:.2f} | ADX: {adx_val:.1f} | "
             f"{'⚠️ TRENDING' if trending else '✅ ranging'} | "
             f"hook_up: {hook_up} | hook_dn: {hook_dn}")

    # ── Month reset ──
    month_str = now.strftime('%Y-%m')
    if state['current_month'] != month_str:
        if state['breaker_on']:
            log.info(f"  🔄 New month {month_str} — circuit breaker reset")
        state['current_month'] = month_str
        state['month_pnl']     = 0.0
        state['breaker_on']    = False
        save_state(state)

    # ── Circuit breaker ──
    if state['breaker_on']:
        log.info(f"  ⛔ Circuit breaker ON | month loss: ${state['month_pnl']:.2f} | "
                 f"resumes next month")
        return

    equity = state['equity']

    # ── Close trade helper ──
    def close_trade(exit_price, forced=False):
        direction = state['position']
        ep        = state['entry_price']
        mult      = state['entry_mult']
        notional  = equity * mult
        gross     = (exit_price-ep)/ep if direction=='long' else (ep-exit_price)/ep
        net       = gross - 2*0.001
        pnl       = notional * net
        reason    = 'forced (CB)' if forced else 'trail exit'

        state['equity']      += pnl
        state['month_pnl']   += pnl
        state['peak_equity']  = max(state['peak_equity'], state['equity'])
        dd = (state['peak_equity']-state['equity'])/state['peak_equity']*100

        win = pnl > 0
        state['total_trades'] += 1
        state['wins']         += int(win)
        state['losses']       += int(not win)

        emoji  = '✅' if win else '❌'
        triple = ' ⚡2x' if mult == TRIPLE_MULT else ''
        log.info(f"  {emoji} CLOSED {direction.upper()}{triple} | "
                 f"${ep:.2f} -> ${exit_price:.2f} | "
                 f"pnl: ${pnl:+.2f} ({net*100:+.2f}%) | "
                 f"equity: ${state['equity']:.2f} | DD: {dd:.1f}% | {reason}")

        state['trades'].append({
            'time':         now.isoformat(),
            'direction':    direction,
            'entry_price':  round(ep, 4),
            'exit_price':   round(exit_price, 4),
            'entry_rsi':    state['entry_rsi'],
            'multiplier':   mult,
            'tripled':      mult == TRIPLE_MULT,
            'net_ret_pct':  round(net*100, 3),
            'pnl_usd':      round(pnl, 2),
            'equity':       round(state['equity'], 2),
            'drawdown_pct': round(dd, 2),
            'month_pnl':    round(state['month_pnl'], 2),
            'adx':          round(adx_val, 2),
            'reason':       reason,
        })

        close_order(direction, exit_price)
        state['position']       = None
        state['entry_price']    = None
        state['entry_rsi']      = None
        state['entry_mult']     = None
        state['hook_rsi_level'] = None
        state['waiting_trail']  = False

        if MONTHLY_LOSS_LIMIT and state['month_pnl'] <= -MONTHLY_LOSS_LIMIT:
            log.warning(f"  ⛔ CIRCUIT BREAKER — month loss ${state['month_pnl']:.2f} "
                        f"exceeded ${MONTHLY_LOSS_LIMIT:.0f}")
            state['breaker_on'] = True

        save_state(state)

    # ── Open trade helper ──
    def open_trade(direction, entry_rsi):
        mult     = get_multiplier(direction, entry_rsi)
        notional = equity * mult
        triple   = ' ⚡2x' if mult == TRIPLE_MULT else ''
        log.info(f"  🔔 OPEN {direction.upper()}{triple} | "
                 f"price: ${price:.2f} | RSI: {entry_rsi:.1f} | "
                 f"size: {mult*100:.0f}% (${notional:.2f})")
        execute_order(direction, notional, price)
        state['position']       = direction
        state['entry_price']    = price
        state['entry_rsi']      = round(entry_rsi, 2)
        state['entry_mult']     = mult
        state['hook_rsi_level'] = None
        state['waiting_trail']  = False
        save_state(state)

    # ── Force close if CB just triggered ──
    if MONTHLY_LOSS_LIMIT and state['month_pnl'] <= -MONTHLY_LOSS_LIMIT:
        if state['position']:
            log.warning("  ⛔ Force closing — circuit breaker")
            close_trade(price, forced=True)
        state['breaker_on'] = True
        save_state(state)
        return

    # ── State machine ──
    if state['position'] is None:
        state['waiting_trail']  = False
        state['hook_rsi_level'] = None
        if trending:
            log.info(f"  ADX {adx_val:.1f} > {ADX_TREND_THRESH} — trending market, no new entries")
        elif hook_dn:
            open_trade('short', r_prev)
        elif hook_up:
            open_trade('long', r_prev)
        else:
            log.info("  No signal — flat")

    elif state['position'] == 'long':
        if not state['waiting_trail'] and hook_dn:
            log.info(f"  Upper hook | RSI prev: {r_prev:.1f} | "
                     f"trail target: RSI <= {r_prev - TRAIL_PULLBACK:.1f}")
            state['waiting_trail']  = True
            state['hook_rsi_level'] = r_prev
            save_state(state)
        elif state['waiting_trail']:
            target = state['hook_rsi_level'] - TRAIL_PULLBACK
            log.info(f"  Trailing long | RSI: {rsi:.1f} | need: <= {target:.1f}")
            if rsi <= target:
                close_trade(price)
                if not state['breaker_on']:
                    if trending:
                        log.info(f"  ADX {adx_val:.1f} > {ADX_TREND_THRESH} — staying flat after exit")
                        state['position'] = None
                    else:
                        open_trade('short', r_prev)

    elif state['position'] == 'short':
        if not state['waiting_trail'] and hook_up:
            log.info(f"  Lower hook | RSI prev: {r_prev:.1f} | "
                     f"trail target: RSI >= {r_prev + TRAIL_PULLBACK:.1f}")
            state['waiting_trail']  = True
            state['hook_rsi_level'] = r_prev
            save_state(state)
        elif state['waiting_trail']:
            target = state['hook_rsi_level'] + TRAIL_PULLBACK
            log.info(f"  Trailing short | RSI: {rsi:.1f} | need: >= {target:.1f}")
            if rsi >= target:
                close_trade(price)
                if not state['breaker_on']:
                    if trending:
                        log.info(f"  ADX {adx_val:.1f} > {ADX_TREND_THRESH} — staying flat after exit")
                        state['position'] = None
                    else:
                        open_trade('long', r_prev)

log.info("Strategy loop ready")

2026-05-19 17:49:57 | INFO | Strategy loop ready


## 📊 Status dashboard

In [8]:
def print_status():
    s   = state
    wr  = s['wins']/s['total_trades']*100 if s['total_trades'] else 0
    dd  = (s['peak_equity']-s['equity'])/s['peak_equity']*100 if s['peak_equity'] else 0
    ret = (s['equity']-STARTING_CAPITAL)/STARTING_CAPITAL*100

    print(f"\n{'='*56}")
    print(f"  SOL TDI Bot — {'PAPER' if PAPER_TRADING else 'LIVE'} | Final Config")
    print(f"{'='*56}")
    print(f"  Equity:        ${s['equity']:,.2f}  ({ret:+.2f}%)")
    print(f"  Peak equity:   ${s['peak_equity']:,.2f}")
    print(f"  Drawdown:      {dd:.2f}%")
    print(f"  Position:      {s['position'] or 'flat'}")
    if s['position']:
        mult   = s['entry_mult'] or 1.0
        triple = ' ⚡2x' if mult == TRIPLE_MULT else ''
        if s['waiting_trail']:
            tgt = (s['hook_rsi_level'] - TRAIL_PULLBACK if s['position']=='long'
                   else s['hook_rsi_level'] + TRAIL_PULLBACK)
            trail_str = f"waiting — target {'<=' if s['position']=='long' else '>='} {tgt:.1f}"
        else:
            trail_str = "watching for hook"
        print(f"  Entry price:   ${s['entry_price']:.2f}")
        print(f"  Entry RSI:     {s['entry_rsi']}")
        print(f"  Size:          {mult*100:.0f}%{triple}")
        print(f"  Trail:         {trail_str}")
    print(f"  Trades:        {s['total_trades']}  ({s['wins']}W / {s['losses']}L)  WR: {wr:.1f}%")
    print(f"  Month PnL:     ${s['month_pnl']:+.2f}  (limit: ${MONTHLY_LOSS_LIMIT:.0f})"           if MONTHLY_LOSS_LIMIT else f"  Month PnL:     ${s['month_pnl']:+.2f}")
    print(f"  Breaker:       {'⛔ ON — paused until next month' if s['breaker_on'] else '✅ off'}")
    print(f"  Month:         {s['current_month']}")
    if s['trades']:
        print(f"\n  Last 5 trades:")
        for t in s['trades'][-5:]:
            emoji  = '✅' if t['pnl_usd'] > 0 else '❌'
            triple = ' ⚡' if t.get('tripled') else ''
            print(f"    {emoji} {t['direction'].upper():5s}{triple} | "
                  f"${t['entry_price']:.2f} -> ${t['exit_price']:.2f} | "
                  f"${t['pnl_usd']:+.2f} ({t['net_ret_pct']:+.2f}%) | "
                  f"ADX: {t.get('adx', 0):.1f} | "
                  f"equity: ${t['equity']:,.2f} | {t['time'][:16]}")
    print()

print_status()


  SOL TDI Bot — PAPER | Final Config
  Equity:        $1,000.00  (+0.00%)
  Peak equity:   $1,000.00
  Drawdown:      0.00%
  Position:      flat
  Trades:        0  (0W / 0L)  WR: 0.0%
  Month PnL:     $+0.00  (limit: $125)
  Breaker:       ✅ off
  Month:         None



## ▶️ Start the bot

In [ ]:
log.info("=== BOT STARTING — FINAL CONFIG ===")
log.info(f"Capital: ${STARTING_CAPITAL:,.2f} | Mode: {'PAPER' if PAPER_TRADING else 'LIVE'}")
log.info(f"Trail: {TRAIL_PULLBACK}pts | ADX filter: >{ADX_TREND_THRESH} | "
         f"CB: ${MONTHLY_LOSS_LIMIT}/mo | 2x sizing: RSI<={RSI_TRIPLE_LONG} or >={RSI_TRIPLE_SHORT}")

# Initial poll
run_once()
print_status()

# Schedule every 15 minutes
schedule.every().hour.at(':00').do(run_once)
schedule.every().hour.at(':15').do(run_once)
schedule.every().hour.at(':30').do(run_once)
schedule.every().hour.at(':45').do(run_once)

log.info(f"Scheduler running | log: {LOG_FILE} | state: {STATE_FILE}")
print(f"\nBot running — polling every 15 minutes")
print(f"Log:   {LOG_FILE}")
print(f"State: {STATE_FILE}")
print(f"Press Ctrl+C to stop\n")

try:
    while True:
        schedule.run_pending()
        time.sleep(30)
except KeyboardInterrupt:
    log.info("=== BOT STOPPED ===")
    print("\nStopped.")
    print_status()

2026-05-19 17:50:01 | INFO | === BOT STARTING — FINAL CONFIG ===
2026-05-19 17:50:01 | INFO | Capital: $1,000.00 | Mode: PAPER
2026-05-19 17:50:01 | INFO | Trail: 0.0pts | ADX filter: >30 | CB: $125.0/mo | 2x sizing: RSI<=20 or >=80
2026-05-19 17:50:01 | INFO | ── Poll 2026-05-19 09:50 UTC ──
2026-05-19 17:50:02 | INFO |   RSI: 37.89 | upper: 66.19 | lower: 30.16 | price: $84.75 | ADX: 13.1 | ✅ ranging | hook_up: False | hook_dn: False
2026-05-19 17:50:02 | INFO |   No signal — flat
2026-05-19 17:50:02 | INFO | Scheduler running | log: sol_bot_final.log | state: sol_bot_final_state.json



  SOL TDI Bot — PAPER | Final Config
  Equity:        $1,000.00  (+0.00%)
  Peak equity:   $1,000.00
  Drawdown:      0.00%
  Position:      flat
  Trades:        0  (0W / 0L)  WR: 0.0%
  Month PnL:     $+0.00  (limit: $125)
  Breaker:       ✅ off
  Month:         2026-05


Bot running — polling every 15 minutes
Log:   sol_bot_final.log
State: sol_bot_final_state.json
Press Ctrl+C to stop



2026-05-19 18:00:02 | INFO | ── Poll 2026-05-19 10:00 UTC ──
2026-05-19 18:00:04 | INFO |   RSI: 38.62 | upper: 66.31 | lower: 29.91 | price: $84.77 | ADX: 14.5 | ✅ ranging | hook_up: False | hook_dn: False
2026-05-19 18:00:04 | INFO |   No signal — flat


## 📋 Trade log viewer

In [ ]:
s = load_state()
if not s['trades']:
    print("No completed trades yet.")
else:
    t = pd.DataFrame(s['trades'])
    t['time']        = pd.to_datetime(t['time']).dt.strftime('%m/%d %H:%M')
    t['entry_price'] = t['entry_price'].map('${:.2f}'.format)
    t['exit_price']  = t['exit_price'].map('${:.2f}'.format)
    t['pnl_usd']     = t['pnl_usd'].map('${:+.2f}'.format)
    t['net_ret_pct'] = t['net_ret_pct'].map('{:+.2f}%'.format)
    t['equity']      = t['equity'].map('${:,.2f}'.format)
    t['size']        = t['multiplier'].map(lambda x: f'{x*100:.0f}%' + (' ⚡' if x > 1 else ''))
    print(t[['time','direction','size','entry_price','exit_price',
             'entry_rsi','net_ret_pct','pnl_usd','equity',
             'drawdown_pct','adx','reason']].to_string(index=False))
    wins   = [tr for tr in s['trades'] if tr['pnl_usd'] > 0]
    losses = [tr for tr in s['trades'] if tr['pnl_usd'] <= 0]
    ret    = (s['equity']-STARTING_CAPITAL)/STARTING_CAPITAL*100
    print(f"\nTotal: {len(s['trades'])} | WR: {len(wins)/len(s['trades'])*100:.1f}% | "
          f"Return: {ret:+.2f}% | Equity: ${s['equity']:,.2f}")

## 🔄 Reset state

In [ ]:
CONFIRM_RESET = False   # change to True to wipe state

if CONFIRM_RESET:
    if Path(STATE_FILE).exists():
        Path(STATE_FILE).unlink()
    state = load_state()
    log.info("State reset")
    print("Done — run the bot cell to start fresh.")
else:
    print("Set CONFIRM_RESET = True to wipe state.")